# Création de csv avec les caractéristiques qui nous intéressent

In [1]:
import pandas as pd
from mutagen import File
from pathlib import Path
import librosa
import numpy as np

## Train

In [2]:
train = pd.read_csv("./data/train.csv")
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 35549 entries, 0 to 35548
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   primary_label     35549 non-null  str    
 1   secondary_labels  35549 non-null  str    
 2   type              35549 non-null  str    
 3   latitude          35549 non-null  float64
 4   longitude         35549 non-null  float64
 5   scientific_name   35549 non-null  str    
 6   common_name       35549 non-null  str    
 7   class_name        35549 non-null  str    
 8   inat_taxon_id     35549 non-null  int64  
 9   author            35549 non-null  str    
 10  license           35549 non-null  str    
 11  rating            35549 non-null  float64
 12  url               35549 non-null  str    
 13  filename          35549 non-null  str    
 14  collection        35549 non-null  str    
dtypes: float64(3), int64(1), str(11)
memory usage: 4.1 MB


In [3]:
print(f"Lignes avec un ou plusieurs labels secondaires : {(train.loc[train["secondary_labels"].ne("[]")].shape[0] * 100 / train["secondary_labels"].shape[0]):.2f} %")
print(f"Lignes avec un type : {(train.loc[train["type"].ne("[]")].shape[0] * 100 / train["type"].shape[0]):.2f} %")

Lignes avec un ou plusieurs labels secondaires : 12.30 %
Lignes avec un type : 63.50 %


In [4]:
# Création csv
new_train_audio = train[["filename", "primary_label"]].copy()
new_train_audio = new_train_audio.rename(columns={"primary_label": "species_id"})

def get_duration(path):
    audio = File(path)    
    return float(audio.info.length) # secondes

train_audio_root = Path("./data/train_audio")
new_train_audio["duration"] = new_train_audio.apply(lambda x: get_duration(train_audio_root /  x["filename"]), axis=1)
new_train_audio["filepath"] = new_train_audio["filename"].apply(lambda x: f"{(train_audio_root / x).as_posix()}")

In [5]:
# Extraction des MFCC's pour chaque fichier audio
def get_MFCC(file):
    audio, sample_rate = librosa.load(file, sr=None)
    mfccs_features = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=40)
    mfccs_scaled_features = np.mean(mfccs_features.T, axis=0)
    return mfccs_scaled_features

new_train_audio["MFCC"] = new_train_audio.apply(lambda x: get_MFCC(train_audio_root /  x["filename"]), axis=1)

d:\conda_envs\ml_env\Lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1536
  warnings.warn(
d:\conda_envs\ml_env\Lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1115
  warnings.warn(
d:\conda_envs\ml_env\Lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1152
  warnings.warn(
d:\conda_envs\ml_env\Lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1672
  warnings.warn(
d:\conda_envs\ml_env\Lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1024
  warnings.warn(
d:\conda_envs\ml_env\Lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=279
  warnings.warn(
d:\conda_envs\ml_env\Lib\site-packages\librosa\core\spectrum.py:266: Us

In [6]:
new_train_audio["filename"] = new_train_audio["filename"].apply(lambda x: x.split("/")[1])
new_train_audio.to_csv("./eda_csv/train_audio_metadata.csv", index=False)

## Soundscapes

In [ ]:
soundscapes = pd.read_csv(f"./data/train_soundscapes_labels.csv")
soundscapes.info()

<class 'pandas.DataFrame'>
RangeIndex: 1478 entries, 0 to 1477
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   filename       1478 non-null   str  
 1   start          1478 non-null   str  
 2   end            1478 non-null   str  
 3   primary_label  1478 non-null   str  
dtypes: str(4)
memory usage: 46.3 KB


In [8]:
# Création csv
new_soundscapes = soundscapes[["filename"]].copy()

soundscapes_root = Path("./data/train_soundscapes")
new_soundscapes["filepath"] = new_soundscapes["filename"].apply(lambda x: f"{(soundscapes_root / x).as_posix()}")
new_soundscapes["MFCC"] = new_soundscapes.apply(lambda x: get_MFCC(soundscapes_root /  x["filename"]), axis=1)

In [ ]:
new_soundscapes.to_csv("./eda_csv/soundscapes_metadata.csv", index=False)

: 